# plot_results.ipynb

This notebook loads the aggregated experiment outputs and creates report-ready figures for the ASC trustworthiness study.

Expected input files:
- `outputs/results_master.csv`
- `outputs/summary_by_model.csv`
- `outputs/summary_by_context.csv`
- `outputs/summary_by_dimension.csv`
- `outputs/pairwise_differences.csv`

Each chart is drawn in its own figure so it is easy to export and insert into the report.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

BASE_DIR = Path.cwd()
OUTPUTS_DIR = BASE_DIR / 'outputs'
FIGURES_DIR = OUTPUTS_DIR / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_MASTER_PATH = OUTPUTS_DIR / 'results_master.csv'
SUMMARY_MODEL_PATH = OUTPUTS_DIR / 'summary_by_model.csv'
SUMMARY_CONTEXT_PATH = OUTPUTS_DIR / 'summary_by_context.csv'
SUMMARY_DIMENSION_PATH = OUTPUTS_DIR / 'summary_by_dimension.csv'
PAIRWISE_PATH = OUTPUTS_DIR / 'pairwise_differences.csv'

required_paths = [
    RESULTS_MASTER_PATH,
    SUMMARY_MODEL_PATH,
    SUMMARY_CONTEXT_PATH,
    SUMMARY_DIMENSION_PATH,
    PAIRWISE_PATH,
]

missing = [str(p) for p in required_paths if not p.exists()]
if missing:
    raise FileNotFoundError(
        'Missing required analysis files. Run generate.py, score_toxicity.py, '\
        'factuality_review_template.py, score_fairness.py, and analyze.py first. '\
        f'Missing: {missing}'
    )

results_master = pd.read_csv(RESULTS_MASTER_PATH)
summary_by_model = pd.read_csv(SUMMARY_MODEL_PATH)
summary_by_context = pd.read_csv(SUMMARY_CONTEXT_PATH)
summary_by_dimension = pd.read_csv(SUMMARY_DIMENSION_PATH)
pairwise_differences = pd.read_csv(PAIRWISE_PATH)

print('Loaded:')
print('  results_master:', results_master.shape)
print('  summary_by_model:', summary_by_model.shape)
print('  summary_by_context:', summary_by_context.shape)
print('  summary_by_dimension:', summary_by_dimension.shape)
print('  pairwise_differences:', pairwise_differences.shape)


In [ ]:
def choose_model_summary(df: pd.DataFrame) -> pd.DataFrame:
    if 'record_type' in df.columns:
        asc_only = df[df['record_type'].fillna('') == 'asc'].copy()
        if not asc_only.empty:
            return asc_only
    return df.copy()


def choose_context_summary(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if 'record_type' in out.columns:
        asc_only = out[out['record_type'].fillna('') == 'asc'].copy()
        if not asc_only.empty:
            out = asc_only
    return out


def save_current_figure(filename: str):
    path = FIGURES_DIR / filename
    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches='tight')
    print(f'Saved: {path}')


model_summary = choose_model_summary(summary_by_model)
context_summary = choose_context_summary(summary_by_context)


## Figure 1. Mean toxicity by model

This figure compares average toxicity across models on the ASC dataset.


In [ ]:
tox_col = 'mean_toxicity'
plot_df = model_summary[['model', tox_col]].dropna().sort_values(tox_col, ascending=False)

plt.figure(figsize=(8, 5))
plt.bar(plot_df['model'], plot_df[tox_col])
plt.title('Mean toxicity by model')
plt.xlabel('Model')
plt.ylabel('Mean toxicity')
plt.xticks(rotation=20, ha='right')
save_current_figure('figure_1_mean_toxicity_by_model.png')
plt.show()

plot_df


## Figure 2. Refusal rate by model

This figure summarizes how often each model refused or declined to answer.


In [ ]:
refusal_col = 'mean_refusal_score'
plot_df = model_summary[['model', refusal_col]].dropna().sort_values(refusal_col, ascending=False)

plt.figure(figsize=(8, 5))
plt.bar(plot_df['model'], plot_df[refusal_col])
plt.title('Refusal rate by model')
plt.xlabel('Model')
plt.ylabel('Mean refusal score')
plt.xticks(rotation=20, ha='right')
save_current_figure('figure_2_refusal_rate_by_model.png')
plt.show()

plot_df


## Figure 3. Mean helpfulness by model

This figure compares how helpful the reviewed responses were on average.


In [ ]:
helpfulness_col = 'mean_helpfulness_score'
plot_df = model_summary[['model', helpfulness_col]].dropna().sort_values(helpfulness_col, ascending=False)

plt.figure(figsize=(8, 5))
plt.bar(plot_df['model'], plot_df[helpfulness_col])
plt.title('Mean helpfulness by model')
plt.xlabel('Model')
plt.ylabel('Mean helpfulness score')
plt.xticks(rotation=20, ha='right')
save_current_figure('figure_3_mean_helpfulness_by_model.png')
plt.show()

plot_df


## Figure 4. Factuality label distribution by model

This figure uses the row-level merged data so you can see the proportion of factuality labels by model.


In [ ]:
factual_df = results_master[['model', 'factuality_label']].copy()
factual_df['factuality_label'] = factual_df['factuality_label'].fillna('').astype(str).str.strip()
factual_df = factual_df[factual_df['factuality_label'] != '']

if factual_df.empty:
    print('No factuality labels found in results_master.csv')
else:
    counts = factual_df.groupby(['model', 'factuality_label']).size().unstack(fill_value=0)
    proportions = counts.div(counts.sum(axis=1), axis=0)

    plt.figure(figsize=(9, 6))
    bottom = np.zeros(len(proportions))
    x = np.arange(len(proportions.index))
    for label in proportions.columns:
        values = proportions[label].values
        plt.bar(x, values, bottom=bottom, label=label)
        bottom = bottom + values
    plt.title('Factuality label distribution by model')
    plt.xlabel('Model')
    plt.ylabel('Proportion')
    plt.xticks(x, proportions.index, rotation=20, ha='right')
    plt.legend(title='Factuality label')
    save_current_figure('figure_4_factuality_distribution_by_model.png')
    plt.show()
    proportions


## Figure 5. Context-level toxicity heatmap

This figure helps identify which contexts were more problematic for each model.


In [ ]:
heat_df = context_summary[['model', 'context', 'mean_toxicity']].dropna().copy()

if heat_df.empty:
    print('No context-level toxicity data found.')
else:
    pivot = heat_df.pivot_table(index='context', columns='model', values='mean_toxicity', aggfunc='mean')
    plt.figure(figsize=(9, max(4, 0.5 * len(pivot))))
    plt.imshow(pivot.values, aspect='auto')
    plt.title('Context-level mean toxicity heatmap')
    plt.xlabel('Model')
    plt.ylabel('Context')
    plt.xticks(range(len(pivot.columns)), pivot.columns, rotation=20, ha='right')
    plt.yticks(range(len(pivot.index)), pivot.index)
    plt.colorbar(label='Mean toxicity')
    save_current_figure('figure_5_context_toxicity_heatmap.png')
    plt.show()
    pivot


## Figure 6. Fairness gap in helpfulness by model

This figure uses pairwise differences. Positive values mean the autistic version received a higher helpfulness score than the neurotypical version; negative values mean the opposite.


In [ ]:
candidate_cols = [c for c in pairwise_differences.columns if c.startswith('helpfulness_score_diff_')]

if not candidate_cols:
    print('No helpfulness difference columns found in pairwise_differences.csv')
else:
    diff_col = candidate_cols[0]
    pairs = pairwise_differences.copy()
    if 'pair_status' in pairs.columns:
        pairs = pairs[pairs['pair_status'] == 'complete_pair']
    plot_df = pairs.groupby('model', dropna=False)[diff_col].mean().dropna().sort_values()

    plt.figure(figsize=(8, 5))
    plt.bar(plot_df.index, plot_df.values)
    plt.axhline(0, linewidth=1)
    plt.title('Average fairness gap in helpfulness by model')
    plt.xlabel('Model')
    plt.ylabel(diff_col)
    plt.xticks(rotation=20, ha='right')
    save_current_figure('figure_6_fairness_gap_helpfulness_by_model.png')
    plt.show()
    plot_df.to_frame(name=diff_col)


## Figure 7. Fairness gap in toxicity by model

This figure shows whether toxicity scores differ between autistic and neurotypical versions of the same scenario.


In [ ]:
candidate_cols = [c for c in pairwise_differences.columns if c.startswith('toxicity_diff_')]

if not candidate_cols:
    print('No toxicity difference columns found in pairwise_differences.csv')
else:
    diff_col = candidate_cols[0]
    pairs = pairwise_differences.copy()
    if 'pair_status' in pairs.columns:
        pairs = pairs[pairs['pair_status'] == 'complete_pair']
    plot_df = pairs.groupby('model', dropna=False)[diff_col].mean().dropna().sort_values()

    plt.figure(figsize=(8, 5))
    plt.bar(plot_df.index, plot_df.values)
    plt.axhline(0, linewidth=1)
    plt.title('Average fairness gap in toxicity by model')
    plt.xlabel('Model')
    plt.ylabel(diff_col)
    plt.xticks(rotation=20, ha='right')
    save_current_figure('figure_7_fairness_gap_toxicity_by_model.png')
    plt.show()
    plot_df.to_frame(name=diff_col)


## Figure 8. Mean stereotype score by model

This figure summarizes manual stereotype judgments.


In [ ]:
stereo_col = 'mean_stereotype_score'
plot_df = model_summary[['model', stereo_col]].dropna().sort_values(stereo_col, ascending=False)

plt.figure(figsize=(8, 5))
plt.bar(plot_df['model'], plot_df[stereo_col])
plt.title('Mean stereotype score by model')
plt.xlabel('Model')
plt.ylabel('Mean stereotype score')
plt.xticks(rotation=20, ha='right')
save_current_figure('figure_8_mean_stereotype_score_by_model.png')
plt.show()

plot_df


## Optional inspection tables

These tables are useful for quick checks before writing the report.


In [ ]:
display(model_summary.head(20))
display(context_summary.head(20))
display(summary_by_dimension.head(20))
